# ✈️ Airline Disruption Management System — Phase 1
### Data Pipeline, Disruption Detection & Predictive Analytics

**Project overview**  
This notebook covers Phase 1 of the Agentic Disruption Management System:
- Download real BTS on-time performance data (2022–2024)
- Fetch NOAA weather observations (METAR) for major hub airports
- Clean and join both datasets
- Run the disruption detection engine (classify, score, flag)
- Build and evaluate a delay prediction model (XGBoost)
- Visualise findings in an interactive dashboard

**Data sources**  
| Source | What it provides | Access |
|--------|-----------------|--------|
| [BTS Transtats](https://www.transtats.bts.gov) | Flight delays, cancellations, causes | Free, public |
| [NOAA Aviation Weather](https://aviationweather.gov/api/data/metar) | METAR weather per airport/hour | Free, no key needed |

---

## 0 · Environment Setup
Mount Google Drive so your data persists across sessions, then install any missing packages.

In [ ]:
# Mount Google Drive — all data saves here and survives session resets
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Airline Project/airline-disruption'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Airline Project/airline-disruption


In [ ]:
# Install any packages not in Colab by default
!pip install pyarrow fastparquet xgboost lightgbm plotly --quiet
print('Packages ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.8 MB/s eta 0:00:00
Packages ready.


In [ ]:
# Core imports used throughout the notebook
import os, time, json, zipfile, io, re, warnings
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# All data goes inside Google Drive project folder
DIR_RAW       = Path(PROJECT_DIR) / 'data/raw'
DIR_PROCESSED = Path(PROJECT_DIR) / 'data/processed'
DIR_WEATHER   = Path(PROJECT_DIR) / 'data/weather'
DIR_SUMMARY   = Path(PROJECT_DIR) / 'data/summary'
DIR_MODELS    = Path(PROJECT_DIR) / 'models'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER, DIR_SUMMARY, DIR_MODELS]:
    d.mkdir(parents=True, exist_ok=True)

print('All directories ready.')
print(f'  Raw data  →  {DIR_RAW}')
print(f'  Processed →  {DIR_PROCESSED}')

All directories ready.
  Raw data  →  /content/drive/MyDrive/Airline Project/airline-disruption/data/raw
  Processed →  /content/drive/MyDrive/Airline Project/airline-disruption/data/processed


## 1 · Configuration
Edit these settings to change which airports and date range to analyse.  
The defaults cover **10 major US hub airports, Jan 2022 – Dec 2024** (~4 million flights).

In [ ]:
# ── Target airports (IATA codes) ────────────────────────────────────────
TARGET_AIRPORTS = ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']

# IATA → ICAO mapping for METAR weather fetching
AIRPORT_ICAO = {
    'JFK': 'KJFK', 'ORD': 'KORD', 'ATL': 'KATL', 'LAX': 'KLAX',
    'DFW': 'KDFW', 'SFO': 'KSFO', 'EWR': 'KEWR', 'MIA': 'KMIA',
    'SEA': 'KSEA', 'BOS': 'KBOS'
}

# ── Date range ───────────────────────────────────────────────────────────────
START_YEAR,  START_MONTH = 2022, 1    # Jan 2022 — includes SW meltdown (Dec 2022)
END_YEAR,    END_MONTH   = 2024, 12   # Dec 2024

# ── BTS fields to download ───────────────────────────────────────────────────
BTS_FIELDS = [
    'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
    'Reporting_Airline', 'Flight_Number_Reporting_Airline',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest',   'DestCityName',   'DestState',
    'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
    'TaxiOut', 'TaxiIn', 'WheelsOff', 'WheelsOn',
    'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
    'Cancelled', 'CancellationCode', 'Diverted',
    'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Distance',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
]

print(f'Airports  : {TARGET_AIRPORTS}')
print(f'Date range: {START_YEAR}-{START_MONTH:02d} → {END_YEAR}-{END_MONTH:02d}')
print(f'BTS fields: {len(BTS_FIELDS)} columns')

Airports  : ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']
Date range: 2022-01 → 2024-12
BTS fields: 40 columns


## 2 · Download BTS On-Time Performance Data

BTS blocks automated POST requests to their download form, so we use their **direct PREZIP URLs** instead —
publicly accessible ZIP files served straight from `transtats.bts.gov/PREZIP/`.
No login, no form, no scraping.

If any month fails, the cell also tries the **Kaggle API** as a fallback (requires your Kaggle credentials).

> **Expected time:** ~1–3 min per year of data. Files are cached in Drive — re-runs skip already-downloaded months.

In [ ]:
# ── Option A: Direct BTS PREZIP download (recommended — no login needed) ──────
#
# BTS hosts pre-built ZIP files at a stable URL pattern:
# https://transtats.bts.gov/PREZIP/On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_{YEAR}_{MONTH}.zip

def download_bts_month(year, month):
    """Download one month via the BTS PREZIP direct URL. Returns local path or None."""
    out_path = DIR_RAW / f'BTS_OTP_{year}_{month:02d}.zip'
    if out_path.exists() and out_path.stat().st_size > 10_000:
        print(f'  [cached] {year}-{month:02d}')
        return out_path

    # BTS PREZIP URL — works for Jan 2018 onwards
    url = (
        f'https://transtats.bts.gov/PREZIP/'
        f'On_Time_Marketing_Carrier_On_Time_Performance_'
        f'Beginning_January_2018_{year}_{month}.zip'
    )

    print(f'  [fetch]  {year}-{month:02d} ...', end=' ', flush=True)
    try:
        r = requests.get(url, timeout=120,
            headers={'User-Agent': 'Mozilla/5.0 (airline-research-project)'})
        r.raise_for_status()
        out_path.write_bytes(r.content)
        kb = out_path.stat().st_size // 1024
        print(f'OK ({kb:,} KB)')
        time.sleep(1)   # polite pause
        return out_path
    except Exception as e:
        print(f'FAILED: {e}')
        # Clean up empty file if created
        if out_path.exists() and out_path.stat().st_size < 1000:
            out_path.unlink()
        return None


print('=== Downloading BTS data (PREZIP direct URLs) ===')
print(f'Range: {START_YEAR}-{START_MONTH:02d} → {END_YEAR}-{END_MONTH:02d}')
print()

downloaded, failed = [], []
for year in range(START_YEAR, END_YEAR + 1):
    m_start = START_MONTH if year == START_YEAR else 1
    m_end   = END_MONTH   if year == END_YEAR   else 12
    for month in range(m_start, m_end + 1):
        p = download_bts_month(year, month)
        (downloaded if p else failed).append(f'{year}-{month:02d}')

print(f'\n✅ Downloaded/cached: {len(downloaded)}')
if failed:
    print(f'❌ Failed: {failed}')
    print('   → Run the Kaggle fallback cell below for the failed months.')

=== Downloading BTS data (PREZIP direct URLs) ===
Range: 2022-01 → 2024-12

  [cached] 2022-01
  [cached] 2022-02
  [cached] 2022-03
  [cached] 2022-04
  [cached] 2022-05
  [cached] 2022-06
  [cached] 2022-07
  [cached] 2022-08
  [cached] 2022-09
  [cached] 2022-10
  [cached] 2022-11
  [cached] 2022-12
  [cached] 2023-01
  [cached] 2023-02
  [cached] 2023-03
  [cached] 2023-04
  [cached] 2023-05
  [cached] 2023-06
  [cached] 2023-07
  [cached] 2023-08
  [cached] 2023-09
  [cached] 2023-10
  [cached] 2023-11
  [cached] 2023-12
  [cached] 2024-01
  [cached] 2024-02
  [cached] 2024-03
  [cached] 2024-04
  [cached] 2024-05
  [cached] 2024-06
  [cached] 2024-07
  [cached] 2024-08
  [cached] 2024-09
  [cached] 2024-10
  [cached] 2024-11
  [cached] 2024-12

✅ Downloaded/cached: 36


In [ ]:
# ── Option B: Kaggle API fallback ────────────────────────────────────────────
# Only run this if Option A failed for some months.
#
# Setup (one-time):
#   1. Go to https://www.kaggle.com → Account → API → Create New Token
#   2. Upload the downloaded kaggle.json when prompted below
#
# Dataset used: daryaheyko/airline-on-time-statistics-and-delay-causes-bts
# Contains BTS OTP data from 2003–2025, same columns as PREZIP files.

USE_KAGGLE_FALLBACK = False  # ← set to True if Option A failed

if USE_KAGGLE_FALLBACK:
    from google.colab import files
    import os

    # Upload kaggle.json credentials
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

    # Install kaggle CLI and download
    !pip install kaggle --quiet
    !kaggle datasets download -d daryaheyko/airline-on-time-statistics-and-delay-causes-bts \
        -p {str(DIR_RAW)} --unzip

    print('Kaggle dataset downloaded to:', DIR_RAW)
else:
    print('Kaggle fallback not enabled. Set USE_KAGGLE_FALLBACK = True if needed.')

Kaggle fallback not enabled. Set USE_KAGGLE_FALLBACK = True if needed.


### ⚠️ If both options above fail — manual download

1. Go to: **https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ**
2. Select your year and month, tick the fields listed in Section 1 (`BTS_FIELDS`)
3. Click **Download** — you get a ZIP file
4. Upload it to: `My Drive → airline-disruption → data → raw`
5. Name it: `BTS_OTP_YYYY_MM.zip` (e.g. `BTS_OTP_2023_06.zip`)
6. Re-run Section 3 (Load and Clean) — it picks up all ZIPs automatically

Alternatively, the **Kaggle dataset below** has the same data pre-packaged:  
https://www.kaggle.com/datasets/daryaheyko/airline-on-time-statistics-and-delay-causes-bts  
https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024

## 3 · Load and Clean BTS Data
Extract CSVs from ZIP files, concatenate, filter to target airports, and engineer features.

In [ ]:
# ── Corrected KEEP_COLS using actual PREZIP column names ─────────────────────
KEEP_COLS = [
    'Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
    'IATA_Code_Marketing_Airline',          # carrier code (AA, DL, WN etc.)
    'Flight_Number_Marketing_Airline',
    'Operating_Airline ',                   # note: trailing space — BTS quirk
    'IATA_Code_Operating_Airline',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest',   'DestCityName',   'DestState',
    'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
    'TaxiOut', 'TaxiIn',
    'CRSArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
    'Cancelled', 'CancellationCode', 'Diverted',
    'Distance', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
]

def process_one_zip(zf_path):
    try:
        with zipfile.ZipFile(zf_path, 'r') as z:
            csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
            with z.open(csv_name) as f:
                # Strip column names first, then match
                raw_peek = pd.read_csv(f, nrows=0, low_memory=False)
                raw_cols = {c.strip(): c for c in raw_peek.columns}  # stripped → original
                use_cols = [raw_cols[k.strip()] for k in KEEP_COLS if k.strip() in raw_cols]

        with zipfile.ZipFile(zf_path, 'r') as z:
            csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f, usecols=use_cols, low_memory=False)
    except Exception as e:
        print(f'  [error] {zf_path.name}: {e}')
        return None

    # Strip ALL column names to remove trailing spaces
    df.columns = df.columns.str.strip()

    # Rename carrier columns to clean standard names
    df = df.rename(columns={
        'IATA_Code_Marketing_Airline':   'Airline',
        'Flight_Number_Marketing_Airline': 'FlightNumber',
        'Operating_Airline':             'OperatingAirline',
        'IATA_Code_Operating_Airline':   'OperatingAirlineCode',
    })

    # Filter to target airports
    mask = df['Origin'].isin(TARGET_AIRPORTS) | df['Dest'].isin(TARGET_AIRPORTS)
    df = df[mask].copy()
    return df if len(df) > 0 else None


def clean_chunk(df):
    df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')
    df = df.dropna(subset=['FlightDate'])

    num_cols = [
        'DepDelay','DepDelayMinutes','ArrDelay','ArrDelayMinutes',
        'CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay',
        'TaxiOut','TaxiIn','AirTime','Distance',
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in ['Cancelled','Diverted','DepDel15','ArrDel15']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    if 'CancellationCode' in df.columns:
        df['CancellationCode'] = df['CancellationCode'].str.strip().replace('', np.nan)
        df['CancellationReason'] = df['CancellationCode'].map(
            {'A':'Carrier','B':'Weather','C':'NAS','D':'Security'})

    cause_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
    for col in cause_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df['PrimaryDelayCause'] = df[cause_cols].idxmax(axis=1).where(
        df[cause_cols].max(axis=1) > 0, other=np.nan
    ).map({
        'CarrierDelay':'Carrier','WeatherDelay':'Weather',
        'NASDelay':'NAS','SecurityDelay':'Security','LateAircraftDelay':'Late Aircraft'
    })

    def severity(row):
        if row['Cancelled'] == 1:       return 'Cancelled'
        d = row.get('ArrDelayMinutes', 0) or 0
        if d >= 120: return 'Severe'
        if d >= 45:  return 'Significant'
        if d >= 15:  return 'Minor'
        return 'On Time'
    df['SeverityTier'] = df.apply(severity, axis=1)

    df['ScheduledDepHour'] = pd.to_numeric(
        df['CRSDepTime'].astype(str).str.zfill(4).str[:2], errors='coerce'
    )
    df['Season'] = df['Month'].map({
        12:'Winter',1:'Winter',2:'Winter',
        3:'Spring',4:'Spring',5:'Spring',
        6:'Summer',7:'Summer',8:'Summer',
        9:'Fall',10:'Fall',11:'Fall'
    })
    df['IsWeekend'] = df['DayOfWeek'].isin([6,7]).astype(int)

    # Downcast to save RAM
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')

    # Year needs more range than int8 (max 127) — use int16 instead
    if 'Year' in df.columns:
        df['Year'] = df['Year'].astype('int16')

    for col in ['Month','DayofMonth','DayOfWeek',
                'DepDel15','ArrDel15','Cancelled','Diverted','IsWeekend']:
        if col in df.columns:
            df[col] = df[col].astype('int8')

    return df.reset_index(drop=True)


# ── Re-process all ZIPs with corrected column names ──────────────────────────
print('Re-processing all ZIPs with corrected column names...\n')

zip_files  = sorted(DIR_RAW.glob('BTS_OTP_*.zip'))
out_path   = DIR_PROCESSED / 'bts_cleaned.parquet'

# Delete old parquet first to start fresh
if out_path.exists():
    out_path.unlink()
    print('Old parquet deleted — rebuilding fresh.\n')

total_rows = 0
for i, zf in enumerate(zip_files):
    chunk = process_one_zip(zf)
    if chunk is None:
        continue
    chunk = clean_chunk(chunk)
    total_rows += len(chunk)

    if i == 0:
        chunk.to_parquet(out_path, index=False, engine='fastparquet')
    else:
        chunk.to_parquet(out_path, index=False, engine='fastparquet', append=True)

    print(f'  [{i+1:02d}/{len(zip_files)}] {zf.name}: {len(chunk):,} rows  '
          f'(running total: {total_rows:,})')
    del chunk

print(f'\n✅ Done. Total rows: {total_rows:,}')
print(f'Saved: {out_path}')

# Reload and verify
df = pd.read_parquet(out_path)
print(f'\nFinal shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Re-processing all ZIPs with corrected column names...

Old parquet deleted — rebuilding fresh.

  [01/36] BTS_OTP_2022_01.zip: 275,486 rows  (running total: 275,486)
  [02/36] BTS_OTP_2022_02.zip: 252,558 rows  (running total: 528,044)
  [03/36] BTS_OTP_2022_03.zip: 284,959 rows  (running total: 813,003)
  [04/36] BTS_OTP_2022_04.zip: 283,691 rows  (running total: 1,096,694)
  [05/36] BTS_OTP_2022_05.zip: 297,709 rows  (running total: 1,394,403)
  [06/36] BTS_OTP_2022_06.zip: 293,584 rows  (running total: 1,687,987)
  [07/36] BTS_OTP_2022_07.zip: 299,820 rows  (running total: 1,987,807)
  [08/36] BTS_OTP_2022_08.zip: 299,291 rows  (running total: 2,287,098)
  [09/36] BTS_OTP_2022_09.zip: 283,206 rows  (running total: 2,570,304)
  [10/36] BTS_OTP_2022_10.zip: 288,520 rows  (running total: 2,858,824)
  [11/36] BTS_OTP_2022_11.zip: 273,728 rows  (running total: 3,132,552)
  [12/36] BTS_OTP_2022_12.zip: 276,648 rows  (running total: 3,409,200)
  [13/36] BTS_OTP_2023_01.zip: 275,102 rows  (

## 4 · Exploratory Analysis
Sanity checks and key statistics before building the detection engine.

In [ ]:
# Fix FlightDate type after parquet reload, then run overview
df = pd.read_parquet(DIR_PROCESSED / 'bts_cleaned.parquet')
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')

# Dataset overview
total     = len(df)
cancelled = df['Cancelled'].sum()
dep_del   = df['DepDel15'].sum()
arr_del   = df['ArrDel15'].sum()

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Flights       : {total:,}')
print(f'  Date range    : {df["FlightDate"].min().date()} → {df["FlightDate"].max().date()}')
print(f'  Airlines      : {df["Airline"].nunique()}')
print(f'  Cancellation  : {cancelled/total*100:.2f}%  ({cancelled:,})')
print(f'  Dep delay≥15m : {dep_del/total*100:.2f}%  ({dep_del:,})')
print(f'  Arr delay≥15m : {arr_del/total*100:.2f}%  ({arr_del:,})')
print()
print('Severity breakdown:')
print(df['SeverityTier'].value_counts().to_string())

  DATASET OVERVIEW
  Flights       : 10,504,936
  Date range    : 2022-01-01 → 2024-12-31
  Airlines      : 10
  Cancellation  : 1.81%  (190,145)
  Dep delay≥15m : 20.04%  (2,105,664)
  Arr delay≥15m : 20.45%  (2,148,534)

Severity breakdown:
SeverityTier
On Time        8166257
Minor          1140970
Significant     697812
Severe          309752
Cancelled       190145


In [ ]:
# Plot 1: Severity distribution by airport
sev_order = ['On Time', 'Minor', 'Significant', 'Severe', 'Cancelled']
sev_colors = {
    'On Time':'#1D9E75', 'Minor':'#FAC775',
    'Significant':'#EF9F27', 'Severe':'#E24B4A', 'Cancelled':'#A32D2D'
}

sev_data = (
    df.groupby(['Origin','SeverityTier'])
    .size().reset_index(name='Count')
)
sev_data['SeverityTier'] = pd.Categorical(sev_data['SeverityTier'], categories=sev_order, ordered=True)
sev_data = sev_data.sort_values('SeverityTier')

fig = px.bar(
    sev_data, x='Origin', y='Count', color='SeverityTier',
    color_discrete_map=sev_colors,
    title='Flight Severity Distribution by Airport',
    labels={'Count':'Number of Flights', 'Origin':'Airport'},
    barmode='stack'
)
fig.update_layout(legend_title='Severity', template='plotly_white')
fig.show()

In [ ]:
# Plot 2: Monthly on-time performance trend
monthly = (
    df.groupby(['Year','Month','Airline'])
    .agg(
        Flights   =('FlightDate','count'),
        Cancelled =('Cancelled','sum'),
        ArrDel15  =('ArrDel15','sum'),
        AvgArrDelay=('ArrDelayMinutes','mean')
    ).reset_index()
)
monthly['YearMonth']    = pd.to_datetime(monthly[['Year','Month']].assign(Day=1))
monthly['CancelPct']    = monthly['Cancelled'] / monthly['Flights'] * 100
monthly['DelayRate']    = monthly['ArrDel15']  / monthly['Flights'] * 100

# Aggregate across all airlines
agg = monthly.groupby('YearMonth').agg(
    CancelPct=('CancelPct','mean'),
    DelayRate=('DelayRate','mean'),
    AvgArrDelay=('AvgArrDelay','mean')
).reset_index()

fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Arrival delay rate (≥15 min)', 'Average arrival delay (minutes)'))
fig2.add_trace(go.Scatter(x=agg['YearMonth'], y=agg['DelayRate'],
    mode='lines+markers', name='Delay rate %', line=dict(color='#E24B4A')), row=1, col=1)
fig2.add_trace(go.Scatter(x=agg['YearMonth'], y=agg['AvgArrDelay'],
    mode='lines+markers', name='Avg delay (min)', line=dict(color='#185FA5')), row=2, col=1)
fig2.update_layout(title='Monthly OTP Trend — All Target Airports',
    template='plotly_white', height=500)
fig2.show()

In [ ]:
# Plot 3: Delay cause breakdown (pie chart)
cause_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
cause_labels = ['Carrier','Weather','NAS (ATC)','Security','Late Aircraft']
cause_totals = df[cause_cols].sum().values

fig3 = px.pie(
    values=cause_totals, names=cause_labels,
    title='Total Delay Minutes by Cause (All Airports)',
    color_discrete_sequence=['#185FA5','#1D9E75','#EF9F27','#A32D2D','#7F77DD']
)
fig3.update_traces(textposition='inside', textinfo='percent+label')
fig3.show()

## 5 · Fetch NOAA METAR Weather Data
Fetch hourly weather observations for each target airport and join them to the BTS data.  
This enriches each flight record with actual weather conditions at departure time.

> **Note:** The NOAA API only returns the past 15 days per call. For historical data,  
> we loop through date ranges in batches. Results are cached in Drive.

In [ ]:
NOAA_URL  = 'https://aviationweather.gov/api/data/metar'
BATCH_DAYS = 14
WX_START  = datetime(START_YEAR, START_MONTH, 1)
WX_END    = datetime(END_YEAR, END_MONTH, 28)
METAR_RAW = DIR_WEATHER / 'metar_raw'
METAR_RAW.mkdir(exist_ok=True)


def fetch_metar_batch(icao, start, end):
    params = {
        'ids':    icao,
        'format': 'json',
        'hours':  int((end - start).total_seconds() / 3600),
        'date':   end.strftime('%Y%m%d_%H%M'),
    }
    try:
        r = requests.get(NOAA_URL, params=params,
            headers={'User-Agent': 'airline-disruption-research'}, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'    [warn] {icao} {start.date()}: {e}')
        return []


print('Fetching METAR data...')
all_wx = []

for iata, icao in AIRPORT_ICAO.items():
    print(f'  {iata} ({icao})')
    cursor = WX_START
    while cursor < WX_END:
        batch_end  = min(cursor + timedelta(days=BATCH_DAYS), WX_END)
        cache_file = METAR_RAW / f"{icao}_{cursor.strftime('%Y%m%d')}_{batch_end.strftime('%Y%m%d')}.json"

        if cache_file.exists():
            with open(cache_file) as f:
                recs = json.load(f)
        else:
            recs = fetch_metar_batch(icao, cursor, batch_end)
            with open(cache_file, 'w') as f:
                json.dump(recs, f)
            time.sleep(0.7)

        for rec in recs:
            rec['IATA'] = iata
        all_wx.extend(recs)
        cursor = batch_end + timedelta(days=1)

wx_raw = pd.DataFrame(all_wx)
print(f'\nTotal METAR records: {len(wx_raw):,}')
wx_raw.head(2)

Fetching METAR data...
  JFK (KJFK)
    [warn] KJFK 2022-01-01: 400 Client Error: Bad Request for url: https://aviationweather.gov/api/data/metar?ids=KJFK&format=json&hours=336&date=20220115_0000
    [warn] KJFK 2022-01-16: 400 Client Error: Bad Request for url: https://aviationweather.gov/api/data/metar?ids=KJFK&format=json&hours=336&date=20220130_0000
    [warn] KJFK 2022-01-31: 400 Client Error: Bad Request for url: https://aviationweather.gov/api/data/metar?ids=KJFK&format=json&hours=336&date=20220214_0000
    [warn] KJFK 2022-02-15: 400 Client Error: Bad Request for url: https://aviationweather.gov/api/data/metar?ids=KJFK&format=json&hours=336&date=20220301_0000
    [warn] KJFK 2022-03-02: 400 Client Error: Bad Request for url: https://aviationweather.gov/api/data/metar?ids=KJFK&format=json&hours=336&date=20220316_0000
    [warn] KJFK 2022-03-17: 400 Client Error: Bad Request for url: https://aviationweather.gov/api/data/metar?ids=KJFK&format=json&hours=336&date=20220331_0000
    

KeyboardInterrupt: 

In [ ]:
def clean_metars(df):
    df = df.copy()
    df['ObsTime'] = pd.to_datetime(
        df.get('reportTime', df.get('obsTime', None)), errors='coerce', utc=True
    )
    df = df.dropna(subset=['ObsTime'])

    df['TempC']       = pd.to_numeric(df.get('temp'),  errors='coerce')
    df['WindSpeedKt'] = pd.to_numeric(df.get('wspd'),  errors='coerce')
    df['WindGustKt']  = pd.to_numeric(df.get('wgst'),  errors='coerce')
    df['VisibSM']     = pd.to_numeric(df.get('visib'), errors='coerce')

    # Classify flight category
    def flight_cat(row):
        vis = row.get('VisibSM', None)
        try: vis = float(vis)
        except: return 'Unknown'
        if vis < 1:  return 'LIFR'
        if vis < 3:  return 'IFR'
        if vis < 5:  return 'MVFR'
        return 'VFR'
    df['FlightCategory'] = df.apply(flight_cat, axis=1)

    # Disruption flags
    df['IsLowVis']   = (df['VisibSM']    <  3).astype(int)
    df['IsHighWind'] = (df['WindSpeedKt']>= 25).astype(int)
    df['IsGust']     = (df['WindGustKt'] >= 35).astype(int)
    df['IsFogOrIFR'] = df['FlightCategory'].isin(['IFR','LIFR']).astype(int)

    df['Date']    = df['ObsTime'].dt.date
    df['DepHour'] = df['ObsTime'].dt.hour

    keep = ['IATA','Date','DepHour','TempC','WindSpeedKt','WindGustKt',
            'VisibSM','FlightCategory','IsLowVis','IsHighWind','IsGust','IsFogOrIFR']
    return df[[c for c in keep if c in df.columns]].reset_index(drop=True)


wx = clean_metars(wx_raw)
wx_path = DIR_WEATHER / 'metar_clean.parquet'
wx.to_parquet(wx_path, index=False)
print(f'Clean METAR records: {len(wx):,}')
print(f'Saved: {wx_path}')
wx['FlightCategory'].value_counts()

In [ ]:
# Join BTS + weather on Origin airport, date, departure hour
df['Date']    = df['FlightDate'].dt.date
df['DepHour'] = df['ScheduledDepHour'].astype('Int64')
wx['DepHour'] = wx['DepHour'].astype('Int64')

df_wx = df.merge(
    wx.rename(columns={'IATA': 'Origin'}),
    on=['Origin', 'Date', 'DepHour'],
    how='left'
)

matched = df_wx['FlightCategory'].notna().sum()
print(f'Flights matched to weather: {matched:,} / {len(df_wx):,} ({matched/len(df_wx)*100:.1f}%)')

enriched_path = DIR_PROCESSED / 'bts_with_weather.parquet'
df_wx.to_parquet(enriched_path, index=False)
print(f'Saved enriched dataset: {enriched_path}')

## 6 · Disruption Detection Engine
This is the core of Phase 1. For each flight, the engine outputs:
- **DisruptionType** — what kind of disruption (weather, carrier, NAS, cascading)
- **SeverityScore** — numeric 0–100 for dashboard display
- **IsDisruption** — binary flag for the prediction model
- **CascadeRisk** — whether a delay is likely to propagate to connected flights

In [ ]:
def run_detection_engine(df):
    """Classify and score every flight record for disruption severity."""
    df = df.copy()

    # ── Disruption type ──────────────────────────────────────────────────────
    def classify_disruption(row):
        if row['Cancelled'] == 1:
            reason = row.get('CancellationReason', 'Unknown')
            return f'Cancellation — {reason}'
        delay = row.get('ArrDelayMinutes', 0) or 0
        if delay < 15:
            return 'None'
        # Dominant cause
        causes = {
            'Weather':      row.get('WeatherDelay', 0) or 0,
            'Carrier':      row.get('CarrierDelay', 0) or 0,
            'NAS':          row.get('NASDelay', 0) or 0,
            'Late Aircraft':row.get('LateAircraftDelay', 0) or 0,
            'Security':     row.get('SecurityDelay', 0) or 0,
        }
        top = max(causes, key=causes.get)
        return f'Delay — {top}' if causes[top] > 0 else 'Delay — Unknown'

    df['DisruptionType'] = df.apply(classify_disruption, axis=1)

    # ── Severity score (0–100) ───────────────────────────────────────────────
    # Scoring weights: arrival delay (main), weather conditions, cancellation penalty
    def score(row):
        if row['Cancelled'] == 1:
            return 100
        delay = row.get('ArrDelayMinutes', 0) or 0
        s = min(delay / 3, 80)   # 240 min delay → 80 points

        # Weather penalty
        cat = row.get('FlightCategory', 'VFR')
        s += {'LIFR': 20, 'IFR': 12, 'MVFR': 5, 'VFR': 0}.get(cat, 0)

        # High wind bonus
        if row.get('IsHighWind', 0): s += 5

        return round(min(s, 100), 1)

    df['SeverityScore']  = df.apply(score, axis=1)
    df['IsDisruption']   = (df['SeverityScore'] >= 15).astype(int)

    # ── Cascade risk flag ────────────────────────────────────────────────────
    # Late aircraft delays are propagation signals — the aircraft is late inbound
    df['CascadeRisk'] = (
        (df.get('LateAircraftDelay', 0) >= 30) |
        (df['SeverityScore'] >= 60)
    ).astype(int)

    return df


print('Running disruption detection engine...')
df_wx = pd.read_parquet(DIR_PROCESSED / 'bts_with_weather.parquet')
df_detected = run_detection_engine(df_wx)

detected_path = DIR_PROCESSED / 'bts_detected.parquet'
df_detected.to_parquet(detected_path, index=False)

print(f'\nDisruption detection complete.')
print(f'  Flights flagged as disruptions: {df_detected["IsDisruption"].sum():,} '
      f'({df_detected["IsDisruption"].mean()*100:.1f}%)')
print(f'  Cascade risk flags: {df_detected["CascadeRisk"].sum():,}')
print()
print('Disruption type breakdown:')
print(df_detected['DisruptionType'].value_counts().head(10).to_string())

In [ ]:
# Visualise: Severity score distribution
fig4 = px.histogram(
    df_detected[df_detected['SeverityScore'] > 0],
    x='SeverityScore', nbins=50, color='SeverityTier',
    color_discrete_map={
        'Minor':'#FAC775', 'Significant':'#EF9F27',
        'Severe':'#E24B4A', 'Cancelled':'#A32D2D'
    },
    title='Severity Score Distribution (Disrupted Flights Only)',
    labels={'SeverityScore':'Severity Score (0–100)'},
    template='plotly_white'
)
fig4.show()

## 7 · Delay Prediction Model (XGBoost)
Train a classifier to predict whether a flight will be delayed (≥15 min arrival delay)  
**before** it departs, using schedule features + weather conditions.

This is the predictive analytics layer that gives airlines early warning.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# ── Feature engineering ──────────────────────────────────────────────────────
FEATURES = [
    'Month', 'DayOfWeek', 'ScheduledDepHour', 'IsWeekend',
    'Distance', 'TaxiOut',
    'WindSpeedKt', 'VisibSM', 'TempC',
    'IsLowVis', 'IsHighWind', 'IsFogOrIFR',
]
TARGET = 'ArrDel15'

# Encode categorical airport + airline features
df_model = df_detected.copy()

for col in ['Origin', 'Dest', 'Reporting_Airline']:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col].fillna('UNK'))
    FEATURES.append(col + '_enc')

# Keep only rows with complete feature + target data
model_df = df_model[FEATURES + [TARGET]].dropna()
model_df[TARGET] = model_df[TARGET].astype(int)

X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {len(X_train):,} rows')
print(f'Test set     : {len(X_test):,} rows')
print(f'Delay rate   : {y.mean()*100:.1f}%')

In [ ]:
# Train XGBoost classifier
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # handle class imbalance
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=20,
    verbosity=0,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Save model
model_path = DIR_MODELS / 'xgb_delay_predictor.json'
model.save_model(str(model_path))
print(f'\nModel saved: {model_path}')

In [ ]:
# Evaluate
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print(f'ROC-AUC: {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['On Time', 'Delayed']))

In [ ]:
# Feature importance plot
importance = pd.Series(model.feature_importances_, index=FEATURES)
importance = importance.sort_values(ascending=True).tail(15)

fig5 = px.bar(
    x=importance.values, y=importance.index,
    orientation='h',
    title='Top 15 Features — XGBoost Delay Predictor',
    labels={'x': 'Feature Importance', 'y': 'Feature'},
    color=importance.values,
    color_continuous_scale='Blues',
    template='plotly_white'
)
fig5.update_layout(showlegend=False)
fig5.show()

## 8 · Operations Dashboard Preview
A summary view of the key KPIs your Phase 1 real-time dashboard will display.  
The full interactive Dash/Streamlit app will be built in the next notebook.

In [ ]:
# KPI summary table for the dashboard
kpi = (
    df_detected.groupby('Origin')
    .agg(
        Total_Flights    = ('FlightDate', 'count'),
        Cancellation_Pct = ('Cancelled',  'mean'),
        Delay_Rate_Pct   = ('ArrDel15',   'mean'),
        Avg_Arr_Delay    = ('ArrDelayMinutes', 'mean'),
        Cascade_Risk_Pct = ('CascadeRisk', 'mean'),
        Avg_Severity     = ('SeverityScore', 'mean'),
    )
    .reset_index()
)
kpi['Cancellation_Pct'] = (kpi['Cancellation_Pct'] * 100).round(2)
kpi['Delay_Rate_Pct']   = (kpi['Delay_Rate_Pct']   * 100).round(2)
kpi['Cascade_Risk_Pct'] = (kpi['Cascade_Risk_Pct'] * 100).round(2)
kpi['Avg_Arr_Delay']    = kpi['Avg_Arr_Delay'].round(1)
kpi['Avg_Severity']     = kpi['Avg_Severity'].round(1)
kpi = kpi.sort_values('Delay_Rate_Pct', ascending=False)

fig6 = go.Figure(data=[go.Table(
    header=dict(
        values=['Airport','Total Flights','Cancel %','Delay Rate %',
                'Avg Arr Delay (min)','Cascade Risk %','Avg Severity Score'],
        fill_color='#1F3864', font=dict(color='white', size=12), align='center'
    ),
    cells=dict(
        values=[kpi[c] for c in kpi.columns],
        fill_color=[['#f0f4ff' if i%2==0 else 'white' for i in range(len(kpi))]],
        align='center', font=dict(size=12)
    )
)])
fig6.update_layout(title='Phase 1 Operations Dashboard — KPI Summary', height=400)
fig6.show()

kpi.to_csv(DIR_SUMMARY / 'kpi_summary.csv', index=False)
print('KPI table saved.')

## 9 · Next Steps

Phase 1 is now complete. Your outputs:

| File | Description |
|------|-------------|
| `bts_cleaned.parquet` | Cleaned BTS flight delay dataset |
| `metar_clean.parquet` | Cleaned NOAA weather observations |
| `bts_with_weather.parquet` | Joined BTS + weather |
| `bts_detected.parquet` | Full dataset with severity scores + disruption flags |
| `xgb_delay_predictor.json` | Trained XGBoost delay prediction model |
| `kpi_summary.csv` | Per-airport KPI table |

**Phase 2 (next notebook):**
- Load `bts_detected.parquet`
- Build the AI decision agent (LLM-powered rebooking logic)
- Generate customer communication messages
- Build the exception router (VIPs, no-alternative flags)
- Deploy the customer-facing dashboard

---
*Airline Disruption Management System — Phase 1 | Built with BTS + NOAA real data*